# Getting started with NISAR Level-2 products

**Goal:** by the end of this notebook you will have downloaded one NISAR GCOV granule,
opened it, understood what is inside it, and exported a georeferenced backscatter
raster that you can stack alongside your existing Sentinel-1/2 layers.

**What you already know that transfers:** reprojection, resampling, quality masking,
time-series alignment. All of that is unchanged.

**What is actually new — only three things:**

| # | New thing | Why it matters |
|---|---|---|
| 1 | Products are **HDF5**, not GeoTIFF | One file contains *all* polarizations + coords + metadata in a directory tree. You navigate it rather than just opening it. |
| 2 | Units are **linear power, gamma-nought** | Sentinel-1 GRD is usually sigma-nought and often already in dB. Mixing them silently corrupts a bandstack. |
| 3 | Data are **covariance terms**, not plain bands | Diagonal terms (`HHHH`, `HVHV`) are your VV/VH analogues. Off-diagonal terms are complex and quad-pol only. |

Run the cells **one at a time**. GCOV granules are large and a couple of steps take minutes.

---

## 0. Setup

### 0a. Install the packages

Run this once. `h5netcdf` is the engine that lets `xarray` read HDF5 groups;
`rioxarray` is what gives you `.rio` for CRS handling and GeoTIFF export.

In [ ]:
# !pip install asf_search earthaccess h5py xarray h5netcdf rioxarray rasterio matplotlib

### 0b. Get a NASA Earthdata Login

Everything gates on this. It is free and takes two minutes.

1. Register at **https://urs.earthdata.nasa.gov/users/new**
2. Create a `~/.netrc` file so the libraries can authenticate without prompting:

```
machine urs.earthdata.nasa.gov login YOUR_USERNAME password YOUR_PASSWORD
```

3. Lock it down — `asf_search` will refuse a world-readable netrc:

```bash
chmod 600 ~/.netrc
```

Searching does **not** need login. Downloading does.

In [3]:
import os, getpass
from pathlib import Path

netrc = Path.home() / ".netrc"
if netrc.exists() and "urs.earthdata.nasa.gov" in netrc.read_text():
    print("Earthdata credentials found in ~/.netrc")
else:
    print("No Earthdata entry found. Create ~/.netrc as described above,")
    print("or uncomment the block below to write it interactively.")

# --- interactive alternative (uncomment to use) ---
# user = input("Earthdata username: ")
# pw   = getpass.getpass("Earthdata password: ")
# with open(netrc, "a") as f:
#     f.write(f"\nmachine urs.earthdata.nasa.gov login {user} password {pw}\n")
# netrc.chmod(0o600)
# print("written")

Earthdata credentials found in ~/.netrc


In [4]:
import os
import numpy as np
import h5py
import xarray as xr
import rioxarray            # registers the .rio accessor on xarray objects
import asf_search as asf
import matplotlib.pyplot as plt

print("asf_search", asf.__version__)
print("xarray    ", xr.__version__)
print("h5py      ", h5py.__version__)

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
asf_search 13.0.1
xarray     2025.4.0
h5py       3.13.0


---
## 1. Before writing any code: look at the coverage

**Do this step in a browser first.** Open **[ASF Vertex](https://search.asf.alaska.edu/)**,
filter the dataset to NISAR, and draw a box over one of your study sites.

You are checking one thing: *does NISAR actually cover my sites, and since when?*
Provisional calibrated L-band products only start from **17 June 2026** acquisitions.
If your field campaign predates that, NISAR cannot be applied retrospectively and the
rest of this notebook is preparation for future seasons rather than present analysis.

Five minutes here can save you a month.

---
## 2. Search for granules

Define your area of interest as a WKT polygon in EPSG:4326 (lon lat order).

In [7]:
AOI = "POLYGON ((-89.74727 29.96523, -89.74727 37.03901, -93.0009 37.03901, -93.0009 29.96523, -89.74727 29.96523))"

START = "2026-05-01"
END   = "2026-08-15"
LEVEL = "GCOV"      # try also: GSLC, GUNW, GOFF, RSLC

In [8]:
results = asf.search(
    dataset=asf.DATASET.NISAR,
    intersectsWith=AOI,
    start=START,
    end=END,
    processingLevel=LEVEL,
    maxResults=50,
)

def granule_size_mb(props):
    """NISAR granules report 'bytes' as {fileName: {'bytes': n, 'format': ...}, ...}
    (one entry per distributed file), unlike other missions where it's a single number.
    Sum across all files to get the total download size."""
    b = props.get("bytes") or 0
    if isinstance(b, dict):
        b = sum((v.get("bytes") or 0) for v in b.values())
    return b / 1e6

print(f"{len(results)} {LEVEL} granules found\n")
for g in results:
    p = g.properties
    size_mb = granule_size_mb(p)
    print(f"{p['fileName']}")
    print(f"    {p['startTime']}  |  {p.get('flightDirection')}"
          f"  |  path {p.get('pathNumber')}  |  {size_mb:,.0f} MB")

50 GCOV granules found

NISAR_L2_PR_GCOV_028_012_D_073_4005_DHDH_A_20260814T010148_20260814T010223_P05023_N_F_J_001.h5
    2026-08-14T01:01:48Z  |  DESCENDING  |  path 12  |  6,638 MB
NISAR_L2_PR_GCOV_028_012_D_072_4005_DHDH_A_20260814T010114_20260814T010149_P05023_N_F_J_001.h5
    2026-08-14T01:01:14Z  |  DESCENDING  |  path 12  |  7,016 MB
NISAR_L2_PR_GCOV_028_012_D_071_4005_DHDH_A_20260814T010040_20260814T010115_P05023_N_F_J_001.h5
    2026-08-14T01:00:40Z  |  DESCENDING  |  path 12  |  6,949 MB
NISAR_L2_PR_GCOV_028_012_D_070_4005_DHDH_A_20260814T010007_20260814T010045_P05023_N_F_J_001.h5
    2026-08-14T01:00:07Z  |  DESCENDING  |  path 12  |  7,689 MB
NISAR_L2_PR_GCOV_028_004_A_020_2005_DHDH_M_20260813T111225_20260813T111259_P05023_N_F_J_001.h5
    2026-08-13T11:12:25Z  |  ASCENDING  |  path 4  |  1,949 MB
NISAR_L2_PR_GCOV_028_004_A_020_4005_DHDH_A_20260813T111225_20260813T111253_P05023_N_F_J_001.h5
    2026-08-13T11:12:25Z  |  ASCENDING  |  path 4  |  5,566 MB
NISAR_L2_PR_GCOV_028

### Reading that output

- **flightDirection** — `ASCENDING` / `DESCENDING`. Backscatter differs systematically
  between the two because the viewing geometry differs. **Do not mix them in one time
  series without accounting for it.** Same discipline you'd apply to Sentinel-1.
- **pathNumber** — the orbit track. A consistent time series should come from one path.
- **size** — GCOV granules are large. Start with one.

> **If you got zero results:** widen the date range, or check the AOI polygon is in
> lon-lat order and actually closes (first vertex == last vertex).

---
## 3. Download one granule

In [10]:
OUT_DIR = "./nisar_data"
os.makedirs(OUT_DIR, exist_ok=True)

session = asf.ASFSession()          # reads ~/.netrc
granule = results[0]                 # just the first one for now

granule.download(path=OUT_DIR, session=session)

H5 = os.path.join(OUT_DIR, granule.properties["fileName"])
print("\nsaved to:", H5)
print("size: %.1f GB" % (os.path.getsize(H5) / 1e9))

KeyboardInterrupt: 

---
## 4. Look inside the file

This is the step that makes HDF5 stop feeling strange.

An HDF5 file is a **container with a directory tree inside it**. Groups are folders,
datasets are arrays. Before writing any analysis code against a new product type,
always print the tree and see what is actually there.

In [11]:
def show_tree(h5_path, max_depth=5):
    """Print the internal directory structure of an HDF5 file."""
    def visit(name, obj):
        depth = name.count("/")
        if depth > max_depth:
            return
        indent = "    " * depth
        leaf = name.split("/")[-1]
        if isinstance(obj, h5py.Dataset):
            print(f"{indent}{leaf}   {obj.shape} {obj.dtype}")
        else:
            print(f"{indent}{leaf}/")
    with h5py.File(h5_path, "r") as f:
        f.visititems(visit)

show_tree(H5)

NameError: name 'H5' is not defined

### The map of a GCOV granule

Buried in that output is the structure you care about:

```
/science/LSAR/GCOV/
    grids/
        frequencyA/                  <-- main band, wider bandwidth. USE THIS.
            HHHH                     <-- backscatter intensity, HH   (real)
            HVHV                     <-- backscatter intensity, HV   (real)
            VVVV, VHVH               <-- if acquired in that pol mode
            HHHV, HHVV, HVVV, ...    <-- complex cross-terms, QUAD-POL ONLY
            xCoordinates             <-- UTM easting,  1-D array
            yCoordinates             <-- UTM northing, 1-D array
            projection               <-- EPSG code lives here
            mask                     <-- layover / shadow / invalid flags
            numberOfLooks
            rtcGammaToSigmaFactor    <-- multiply to convert gamma0 -> sigma0
        frequencyB/                  <-- secondary, narrower band, often absent
    metadata/
        orbit/ attitude/ processingInformation/ radarGrid/
    identification/                  <-- time, track, frame, polarization mode
```

Two naming notes:

- `LSAR` = **L-band**. S-band products use `SSAR` in exactly the same position.
- `frequencyA` vs `frequencyB` are the two bandwidth channels *within* a band —
  not to be confused with L vs S.

---
## 5. Read the acquisition metadata

Do this before trusting any pixel. In particular you want the **polarization mode**,
because it determines which covariance terms actually exist.

In [ ]:
with h5py.File(H5, "r") as f:
    grp = f["/science/LSAR/identification"]
    for k in sorted(grp):
        try:
            v = grp[k][()]
            if isinstance(v, bytes):
                v = v.decode()
            print(f"  {k:38s} {v}")
        except Exception as e:
            print(f"  {k:38s} <{e}>")

In [ ]:
# Which covariance terms are actually present in this granule?
GRID = "/science/LSAR/GCOV/grids/frequencyA"

with h5py.File(H5, "r") as f:
    g = f[GRID]
    print("datasets in frequencyA:\n")
    for k in sorted(g):
        d = g[k]
        if isinstance(d, h5py.Dataset):
            print(f"  {k:28s} {str(d.shape):20s} {d.dtype}")

**Interpreting what you see:**

- Only `HHHH` and `HVHV` present -> dual-pol HH/HV acquisition
- Only `VVVV` and `VHVH` -> dual-pol VV/VH
- All four diagonals **plus** complex off-diagonals -> quad-pol

Most routine NISAR land acquisitions are dual-pol. Quad-pol is used sparingly because
it halves the swath. Don't plan a method around quad-pol until you've confirmed it
exists over your sites.

---
## 6. Load a polarization as a georeferenced array

Now we move from `h5py` (raw access) to `xarray` + `rioxarray` (labelled arrays with a CRS).

This is the function you will actually reuse. Read the comments — the unit conversion
in particular is where people quietly get it wrong.

In [ ]:
GRID = "/science/LSAR/GCOV/grids/frequencyA"

def load_gcov(h5_path, pol="HHHH", to_db=True, apply_mask=True):
    """
    Load one GCOV covariance term as a georeferenced xarray DataArray.

    pol       : 'HHHH' | 'HVHV' | 'VVVV' | 'VHVH'  (diagonal = intensity)
    to_db     : GCOV stores LINEAR POWER in gamma0. Convert to dB so it is
                comparable with the Sentinel-1 layers you already have.
    apply_mask: drop layover / shadow / invalid samples.
    """
    ds = xr.open_dataset(
        h5_path,
        engine="h5netcdf",
        group=GRID,
        phony_dims="sort",
        drop_variables=["listOfCovarianceTerms", "listOfPolarizations"],
    )

    da = ds[pol]

    # Coordinates ship as separate 1-D variables, so attach them explicitly.
    da = da.rename({da.dims[0]: "y", da.dims[1]: "x"})
    da = da.assign_coords(x=ds["xCoordinates"].values,
                          y=ds["yCoordinates"].values)

    # Pull the EPSG code out of the projection variable.
    # NOTE: exact location varies slightly between product versions --
    # if this raises, inspect ds['projection'].attrs and adjust.
    proj = ds["projection"]
    epsg = proj.attrs.get("epsg_code", None)
    if epsg is None:
        epsg = int(proj.values)
    da = da.rio.write_crs(f"EPSG:{int(epsg)}")

    if apply_mask:
        # Non-zero flags layover/shadow/invalid. Check mask.attrs for the exact
        # code meanings in your product version before relying on this.
        da = da.where(ds["mask"].values == 0)

    if to_db:
        da = 10 * np.log10(da.where(da > 0))
        da.attrs["units"] = "dB (gamma0)"
    else:
        da.attrs["units"] = "linear power (gamma0)"

    da.attrs["polarization"] = pol
    return da

In [ ]:
hh = load_gcov(H5, pol="HHHH")

print(hh)
print()
print("CRS:  ", hh.rio.crs)
print("shape:", hh.shape)
print("pixel size:", hh.rio.resolution())
print("bounds:", hh.rio.bounds())

> **If the CRS line raised an exception**, run `xr.open_dataset(...)['projection'].attrs`
> and look at what's actually there. This is the one field most likely to have moved
> between product versions. Everything else in the loader is stable.

### Pixel size sanity check

GCOV posting depends on the **acquisition bandwidth**, so it is not constant across granules:

| Bandwidth | Posting |
|---|---|
| 40 MHz | 10 m |
| 20 or 77 MHz | 20 m |
| 5 MHz | 80 m |

Never hardcode the resolution. Read it per granule, as above.

---
## 7. Quick look

Subsample before plotting — these arrays are large.

In [ ]:
stride = 10
small = hh[::stride, ::stride]

fig, ax = plt.subplots(figsize=(9, 9))
small.plot.imshow(ax=ax, cmap="gray", robust=True,
                  cbar_kwargs={"label": hh.attrs["units"]})
ax.set_title(f"NISAR GCOV {hh.attrs['polarization']}")
ax.set_aspect("equal")
plt.show()

**What to look for in a rice landscape:**

- **Very dark** — open water, smooth and specular, reflecting energy away from the sensor.
  Flooded paddy before canopy development.
- **Bright** — double-bounce from the water/stem interaction. This is the flooded-vegetation
  signature, and it is *the* reason L-band is worth the trouble: at C-band this signal is
  largely gone once the canopy closes.
- **Mid-grey, textured** — dry soil, developed canopy volume scattering, urban.

---
## 8. Compare polarizations

The HH/HV (or VV/VH) contrast carries structural information. Cross-pol is generated
mostly by volume scattering inside a canopy, so it tracks biomass; co-pol is more
sensitive to surface and double-bounce geometry.

In [ ]:
# Adjust these to match the pols actually present in your granule (from section 5)
POL_CO, POL_CROSS = "HHHH", "HVHV"

co    = load_gcov(H5, pol=POL_CO)
cross = load_gcov(H5, pol=POL_CROSS)
ratio = co - cross          # in dB, a ratio is a difference

fig, axes = plt.subplots(1, 3, figsize=(19, 6.5))
for ax, da, title in zip(
    axes,
    [co, cross, ratio],
    [f"co-pol ({POL_CO})", f"cross-pol ({POL_CROSS})", "co/cross ratio (dB)"],
):
    da[::stride, ::stride].plot.imshow(ax=ax, cmap="gray", robust=True)
    ax.set_title(title)
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()

---
## 9. Export to GeoTIFF

Now it is an ordinary raster. Open it in QGIS next to a Sentinel-1 scene of the same
area — seeing them side by side teaches more than any documentation.

In [ ]:
co.rio.to_raster("nisar_HH_db.tif", driver="COG", compress="DEFLATE")
cross.rio.to_raster("nisar_HV_db.tif", driver="COG", compress="DEFLATE")
print("written")

---
## 10. Into your bandstack

To append NISAR layers to your existing Sentinel-1/2 stack you need them on a common
grid. `rioxarray` handles this the same way you already do for Sentinel data.

In [ ]:
# Reproject/resample NISAR onto the grid of an existing stack layer.
#
# reference = rioxarray.open_rasterio("my_sentinel_stack.tif")
# nisar_on_grid = co.rio.reproject_match(reference)
#
# stack = xr.concat([reference, nisar_on_grid], dim="band")

print("adapt the above to your existing stack")

### Two gotchas that will bite you

1. **UTM zone changes between frames.** Granules along a track can land in different
   UTM zones. Always reproject to one explicit target CRS rather than assuming they match.
2. **Ascending vs descending.** Keep them as separate features, or model the geometry
   effect. Pooling them blindly injects variance that looks like noise.

---
## 11. The actual science step

Before any machine learning, do this:

1. Extract NISAR backscatter at each of Ben's sensor locations
2. Plot backscatter against measured water level
3. **Stratify by canopy stage** (use your Sentinel-2 NDVI to define the stages)

The question you are really asking is narrow and testable:

> Does L-band backscatter retain sensitivity to water level **after canopy closure**,
> where the C-band features flatten out?

If yes, you have your result, and everything afterwards is engineering. If no, you have
learned something important cheaply, and you have saved yourself building a pipeline
around a signal that isn't there.

A useful diagnostic plot to make: residuals from your *current* C-band model, binned by
NDVI. If error grows sharply as the canopy closes, that is the penetration ceiling made
visible, and it is the single strongest argument for the L-band approach.

In [ ]:
# Sketch — sample NISAR at in-situ sensor points
#
# import pandas as pd
# sensors = pd.read_csv("ben_sensors.csv")     # columns: id, lon, lat, date, water_level_cm
#
# pts = gpd.GeoDataFrame(
#     sensors,
#     geometry=gpd.points_from_xy(sensors.lon, sensors.lat),
#     crs="EPSG:4326",
# ).to_crs(co.rio.crs)
#
# sensors["nisar_hh_db"] = [
#     float(co.sel(x=p.x, y=p.y, method="nearest")) for p in pts.geometry
# ]
#
# sensors.plot.scatter(x="nisar_hh_db", y="water_level_cm", c="ndvi", cmap="viridis")

---
## Where to go next

- **[NISAR Data User Guide (ASF)](https://nisar-docs.asf.alaska.edu/)** — authoritative product reference
- **[NASA Earthdata: Work with NISAR sample data](https://www.earthdata.nasa.gov/learn/tutorials/work-nisar-sample-data)** — official tutorial notebooks
- **[MAAP NISAR access notebook](https://docs.maap-project.org/en/develop/science/NISAR/NISAR_access.html)** — the clearest worked HDF5/xarray example
- **[ASF Vertex](https://search.asf.alaska.edu/)** — search UI
- **[Earthdata Forum](https://forum.earthdata.nasa.gov/)** — where to ask when something in the product doesn't match the spec

### A note on this notebook

The HDF5 paths follow the published NISAR product specification, but this notebook has
not been executed against a live granule. Expect to adjust one or two field names on
first contact — most likely the EPSG extraction in section 6. That is exactly what
`show_tree()` in section 4 is for.